## Challenge Data Preprocess ##

In [1]:
import pandas as pd
import numpy as np
import sys
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.optimize import minimize_scalar
import warnings
import os
import glob
import pickle
sys.path.append('../../map/')
from map import *

In [2]:
def get_object_keygroups(df):
    # Group columns in csv based on each item
    groups = {}
    current_group = []
    current_key = 'Timestamp'

    for col in df.columns:
        if 'xctr' in col:  # Assuming each group starts with 'xctr'
            if current_group:
                groups[current_key] = current_group
            
            current_key = col
            current_group = [col]
        else:
            current_group.append(col)

    # Add the last group
    if current_group:
        groups[current_key] = current_group
    
    groups.pop('Timestamp')
    return groups

In [3]:
object_type_subclass = {
    0: 'Other',
    1: 'Passenger_Vehicle',
    2: 'Vehicle_Other',
    3: 'VRU_Child',
    4: 'VRU_Adult',
    5: 'VRU_Adult_Using_Motorized_Bicycle', 
    6: 'VRU_Adult_Using_Manual_Wheelchair',
    7: 'VRU_Adult_Using_Motorized_Wheelchair',
    8: 'VRU_Adult_Using_Cane',
    9: 'VRU_Adult_Using_Stroller',
    10: 'VRU_Adult_Using_Walker',
    11: 'VRU_Adult_Using_Manual_Bicycle',
    12: 'VRU_Adult_Using_Electric_Scooter',
    13: 'VRU_Adult_Using_Manual_Scooter',
    14: 'VRU_Adult_Using_Skateboard',
    15: 'VRU_Adult_Using_Crutches',
    16: 'VRU_Adult_Using_Cardboard_Box',
    17: 'VRU_Adult_Using_Umbrella',
    18: 'VRU_Other'
}

object_type_class = {
    0: 'Other',
    1: 'Vehicle',
    2: 'VRU'
}

waymo_object_type = {
    0: 'unset', 1: 'vehicle', 2: 'pedestrian', 3: 'cyclist', 4: 'dummy', 5: 'others'
}
subclass_values = list(object_type_subclass.values())

pedestrian_class = ['VRU_Child', 'VRU_Adult','VRU_Adult_Using_Manual_Wheelchair',\
                            'VRU_Adult_Using_Motorized_Wheelchair', 'VRU_Adult_Using_Cane', 'VRU_Adult_Using_Stroller',\
                            'VRU_Adult_Using_Walker', 'VRU_Adult_Using_Skateboard', 'VRU_Adult_Using_Crutches', 'VRU_Adult_Using_Cardboard_Box',\
                            'VRU_Adult_Using_Umbrella']
cyclist_class = ['VRU_Adult_Using_Motorized_Bicycle','VRU_Adult_Using_Manual_Bicycle',\
                'VRU_Adult_Using_Electric_Scooter', 'VRU_Adult_Using_Manual_Scooter']
vehicle_class = ['Passenger_Vehicle', 'Vehicle_Other']

map_class = {
    0: 'Other',
    1: 'Lane',
    2: 'SideWalk',
    3: 'CrossWalk',
    4: 'Boundary',
}

In [4]:
# smoothing parameters 
degree = 5
# frequency of the data
time_disp = 0.1 # 10 Hz, hardcode rn

In [5]:
# smooth 2 segments of the trajectory by optimization
def optimize_pair(x, y, start_index_1, end_index_2, k1, k2, k3, points_per_segment, prev_end=None):

    s1 = np.linspace(0, 1, end_index_2 - start_index_1)
    x_segment = x[start_index_1:end_index_2]
    y_segment = y[start_index_1:end_index_2]
    
    coeffs_x = np.polyfit(s1, x_segment, degree)
    coeffs_y = np.polyfit(s1, y_segment, degree)
    
    def objective(coeffs):
        # Extract the coefficients for x and y
        coeffs_x = coeffs[:degree + 1]
        coeffs_y = coeffs[degree + 1:]
        
        # Define polynomials for x and y
        poly_x = np.poly1d(coeffs_x)
        poly_y = np.poly1d(coeffs_y)
        
        # Define the smoothness objective (e.g., minimizing the integral of the square of the third derivative)
        s = np.linspace(0, 1, len(x_segment))
    
        # Calculate the third derivatives of the polynomials
        fx2 = np.polyder(poly_x, 2)(s)
        fy2 = np.polyder(poly_y, 2)(s)
        fx3 = np.polyder(poly_x, 3)(s)
        fy3 = np.polyder(poly_y, 3)(s)
        # Compute the smoothness criteria
        smoothness = k1 * np.sum(fx3**2 + fy3**2) + k2 * np.sum(fx2**2 + fy2**2)
        
        # Define the closeness to original points for the segment
        segment_x_polyval = np.polyval(poly_x, s)
        segment_y_polyval = np.polyval(poly_y, s)
        original_distance = np.sum((segment_x_polyval - x_segment)**2 + (segment_y_polyval - y_segment)**2)
        
        # Combine objectives with weighting factors
        return smoothness + k3 * original_distance  # The weighting factor can be adjusted
    
    initial_guess = np.hstack([coeffs_x, coeffs_y])
    constraints = []
    
    if prev_end is not None:
        def continuity_constraint(coeffs):
            coeffs_x = coeffs[:degree + 1]
            coeffs_y = coeffs[degree + 1:]
            poly_x = np.poly1d(coeffs_x)
            poly_y = np.poly1d(coeffs_y)
            return [
                poly_x(0) - prev_end[0],
                poly_y(0) - prev_end[1],
                np.polyder(poly_x, 1)(0) - prev_end[2],
                np.polyder(poly_y, 1)(0) - prev_end[3],
                np.polyder(poly_x, 2)(0) - prev_end[4],
                np.polyder(poly_y, 2)(0) - prev_end[5],
                # np.polyder(poly_y, 3)(0) - prev_end[6],
                # np.polyder(poly_y, 4)(0) - prev_end[7],
            ]
        constraints.append({'type': 'eq', 'fun': continuity_constraint})
    


    result = minimize(objective, initial_guess, constraints=constraints)
    
    opt_coeffs_x = result.x[:degree + 1]
    opt_coeffs_y = result.x[degree + 1:]

    return opt_coeffs_x, opt_coeffs_y

In [6]:
# Function to find the closest point on the polynomial to the original point (orig_x, orig_y)
def closest_point_on_polynomial(poly_x, poly_y, orig_x, orig_y):
    # Define the distance function to minimize
    def distance_function(s):
        # Calculate the x and y coordinates on the polynomial at parameter s
        x_poly = np.polyval(poly_x, s)
        y_poly = np.polyval(poly_y, s)
        # Calculate the squared Euclidean distance
        return (x_poly - orig_x)**2 + (y_poly - orig_y)**2

    # Minimize the distance function over the interval [0, 1]
    result = minimize_scalar(distance_function, bounds=(0, 1), method='bounded')

    # Optimal parameter s for the closest point
    optimal_s = result.x

    # Coordinates of the closest point on the polynomial
    closest_x = np.polyval(poly_x, optimal_s)
    closest_y = np.polyval(poly_y, optimal_s)
    
    return closest_x, closest_y, optimal_s

# Optimize the trajectory
def optimize_traj(x, y, k1,k2,k3,points_per_segment):
    '''
    return:
    x_coeff_list: list of coefficients of x polynomials
    y_coeff_list: list of coefficients of y polynomials
    closest_x_points: list of x coordinates of the closest points on the polynomials
    closest_y_points: list of y coordinates of the closest points on the polynomials
    derivatives_x: list of x derivatives at the closest points for heading calculation
    derivatives_y: list of y derivatives at the closest points for heading calculation
    '''
    n_points = len(x)
    num_segments = n_points // points_per_segment
    x_coeff_list = []
    y_coeff_list = []
    # Optimization of each pair of segments
    prev_end = None

    if len(x) < 8:
        return None
    closest_x_points = []
    closest_y_points = []
    derivatives_x = []
    derivatives_y = []
    if num_segments == 1 or num_segments == 0:
        opt_coeffs_x, opt_coeffs_y = optimize_pair(x, y, 0, len(x),k1,k2,k3,points_per_segment)
        x_coeff_list.append(opt_coeffs_x)
        y_coeff_list.append(opt_coeffs_y)
        opt_poly_x = np.poly1d(opt_coeffs_x)
        opt_poly_y = np.poly1d(opt_coeffs_y)
        for j in range(len(x)):
            orig_x = x[j]
            orig_y = y[j]
            closest_x, closest_y, optimal_s = closest_point_on_polynomial(opt_poly_x, opt_poly_y, orig_x, orig_y)
            closest_x_points.append(closest_x)
            closest_y_points.append(closest_y)
            derivatives_x.append(np.polyder(opt_poly_x, 1)(optimal_s))
            derivatives_y.append(np.polyder(opt_poly_y, 1)(optimal_s))
        return x_coeff_list, y_coeff_list, closest_x_points, closest_y_points, derivatives_x, derivatives_y
    


    optimal_s = None
    for i in range(num_segments - 1):
        start_index_1 = i * points_per_segment
        end_index_2 = (i + 2) * points_per_segment if i < num_segments - 2 else len(x)
        
        # Optimize the pair (segment i and segment i+1)
        # opt_coeffs_x, opt_coeffs_y, prev_end = optimize_pair(x, y, start_index_1, end_index_2, prev_end)
        opt_coeffs_x, opt_coeffs_y = optimize_pair(x, y, start_index_1, end_index_2,k1,k2,k3,points_per_segment,prev_end)
        x_coeff_list.append(opt_coeffs_x)
        y_coeff_list.append(opt_coeffs_y)

        # Define the polynomial functions for x and y
        opt_poly_x = np.poly1d(opt_coeffs_x)
        opt_poly_y = np.poly1d(opt_coeffs_y)

        # Find and record the closest points on the polynomial for the original points in this segment
        if end_index_2 == len(x):
            end_index = len(x)
        else:
            end_index = (i + 1) * points_per_segment
        for j in range(start_index_1, end_index):
            orig_x = x[j]
            orig_y = y[j]
            # get the closet point o the polynomial of the original point
            closest_x, closest_y,optimal_s = closest_point_on_polynomial(opt_poly_x, opt_poly_y, orig_x, orig_y)
            closest_x_points.append(closest_x)
            closest_y_points.append(closest_y)
            derivatives_x.append(np.polyder(opt_poly_x, 1)(optimal_s))
            derivatives_y.append(np.polyder(opt_poly_y, 1)(optimal_s))
        prev_end = [opt_poly_x(optimal_s),
                opt_poly_y(optimal_s),
                np.polyder(opt_poly_x, 1)(optimal_s),
                np.polyder(opt_poly_y, 1)(optimal_s),
                np.polyder(opt_poly_x, 2)(optimal_s),
                np.polyder(opt_poly_y, 2)(optimal_s)]
    return x_coeff_list, y_coeff_list, closest_x_points, closest_y_points, derivatives_x, derivatives_y

In [7]:
# get traj info from csv
def decode_tracks_from_csv(df, object_id_start, csv_file):
    csv_keygroup = get_object_keygroups(df)
    track_infos = {
        'object_id': [],  
        'object_type_subclass': [],  
        'object_type_class': [],
        'object_type': [],  # {0: unset, 1: vehicle, 2: pedestrian, 3: cyclist, 4: others}
        'trajs': []
    }

    for key, group in csv_keygroup.items(): # retrieve each object
        # filter out the static vehicle
        if 'Vehicle_Other' in key:
            continue
        # get the state of the object at each timestamp
        # make sure key in order and key exists
        assert 'xctr' in group[0]
        assert 'yctr' in group[1]
        assert 'zctr' in group[2]
        assert 'xlen' in group[3]
        assert 'ylen' in group[4]
        assert 'zlen' in group[5]
        assert 'xrot' in group[6]
        assert 'yrot' in group[7]
        assert 'zrot' in group[8]

        xctr = df[group[0]].values
        yctr = df[group[1]].values
        zctr = df[group[2]].values
        xlen = df[group[3]].values
        ylen = df[group[4]].values
        zlen = df[group[5]].values
        zrot = df[group[8]].values

        # create valid token that is 1 when the vehicle has xyz value
        valid = np.ones(xctr.shape)
        valid[np.isnan(xctr) | np.isnan(yctr) | np.isnan(zctr)] = 0


        dummy_flag = False
        if 'Dummy' in key:
            dummy_flag = True
            key_clean = key.replace('_Dummy', '')
        else:
            key_clean = key

        cur_subclass_clean = '_'.join(key_clean.split('_')[:-1])

        # get the type of the object for static and smoothing threshold and parameters
        pedestrian_class = ['VRU_Child', 'VRU_Adult','VRU_Adult_Using_Manual_Wheelchair',\
                            'VRU_Adult_Using_Motorized_Wheelchair', 'VRU_Adult_Using_Cane', 'VRU_Adult_Using_Stroller',\
                            'VRU_Adult_Using_Walker', 'VRU_Adult_Using_Skateboard', 'VRU_Adult_Using_Crutches', 'VRU_Adult_Using_Cardboard_Box',\
                            'VRU_Adult_Using_Umbrella']
        cyclist_class = ['VRU_Adult_Using_Motorized_Bicycle','VRU_Adult_Using_Manual_Bicycle',\
                        'VRU_Adult_Using_Electric_Scooter', 'VRU_Adult_Using_Manual_Scooter']
        vehicle_class = ['Passenger_Vehicle', 'Vehicle_Other']
        
        temp_type = 'Vehicle'
        if cur_subclass_clean in pedestrian_class:
            temp_type = 'Pedestrian'
        elif cur_subclass_clean in cyclist_class:
            temp_type = 'Cyclist'       

  
        if temp_type == 'Vehicle':
            static_threshold = 0.1
            k1 = 0.0001
            k2 = 0.0001
            k3 = 10
            degree = 5
            points_per_segment = 20
        
        elif temp_type == 'Pedestrian':
            static_threshold = 0.015
            k1 = 0.0001
            k2 = 0.0001
            k3 = 1
            points_per_segment = 40
        
        elif temp_type == 'Cyclist':
            static_threshold = 0.015
            k1 = 0.0001
            k2 = 0.0001
            k3 = 1
            points_per_segment = 40

        # filter out the static points
        valid_x = xctr[valid == 1]
        valid_y = yctr[valid == 1]
        dx = np.diff(valid_x)
        dy = np.diff(valid_y)
        static_mask = (np.abs(dx) <= static_threshold) & (np.abs(dy) <= static_threshold)
        static_start = static_mask[0]
        static_mask = np.concatenate(([static_start], static_mask))
        # Create a mask for non-static parts (static_mask is not True)
        non_static_mask = ~static_mask
        # Filter out the non-static points
        non_static_x = valid_x[non_static_mask]
        non_static_y = valid_y[non_static_mask]

        # optimize the non static points
        optimized_result = optimize_traj(non_static_x, non_static_y, k1,k2,k3,points_per_segment)
        
        # optimization failed since non static points are less than (8? need to check)
        if optimized_result is None:
            print('Optimization failed')
            # print(csv_file)
            # print(key)
            
            # plt.scatter(valid_x[static_mask], valid_y[static_mask], color='blue', label='Static Points')
            # plt.scatter(valid_x[non_static_mask], valid_y[non_static_mask], color='orange', label='Non-static Points')

            print("-------all static--------")
            valid_heading = np.zeros(valid_x.shape)
            # continue
            
            
        else:
            x_coeff_list, y_coeff_list,closest_x_points, closest_y_points, derivative_x, derivative_y = optimized_result
    
            optimized_valid_x = np.full(valid_x.shape, np.nan)
            optimized_valid_y = np.full(valid_y.shape, np.nan)
            optimized_valid_x[non_static_mask] = closest_x_points
            optimized_valid_y[non_static_mask] = closest_y_points

            valid_heading = np.full(valid_x.shape, np.nan)
            valid_heading[non_static_mask] = np.arctan2(derivative_y, derivative_x)


            # Fill in the static points (traj and heading), optimized_valid_x, optimized_valid_y unused now, might be deleted later
            static_indices = np.where(static_mask)[0]
            for i in static_indices:
                if i > 0:
                    # Find the last non-static point before the current static point
                    previous_non_static = i - 1
                    while previous_non_static >= 0 and static_mask[previous_non_static]:
                        previous_non_static -= 1
                    if previous_non_static >= 0:
                        optimized_valid_x[i] = optimized_valid_x[previous_non_static]
                        optimized_valid_y[i] = optimized_valid_y[previous_non_static]
                        valid_heading[i] = valid_heading[previous_non_static]

                if np.isnan(optimized_valid_x[i]):  # If no previous non-static point was found
                    # Use the next non-static point if the static point is at the start or no previous found
                    next_non_static = i + 1
                    while next_non_static < len(static_mask) and static_mask[next_non_static]:
                        next_non_static += 1
                    if next_non_static < len(static_mask):
                        optimized_valid_x[i] = optimized_valid_x[next_non_static]
                        optimized_valid_y[i] = optimized_valid_y[next_non_static]
                        valid_heading[i] = valid_heading[next_non_static]

        # Calculate velocity 
        time_disp = 0.1
        velocity_x_valid = np.zeros(len(valid_x))
        velocity_y_valid = np.zeros(len(valid_y))

        # Compute velocities for all points except the first
        velocity_x_valid[1:] = valid_x[1:] - valid_x[:-1]
        velocity_y_valid[1:] = valid_y[1:] - valid_y[:-1]

        # Assign the velocity of the second point to the first point
        velocity_x_valid[0] = velocity_x_valid[1]
        velocity_y_valid[0] = velocity_y_valid[1]

        velocity_x_valid = velocity_x_valid/time_disp
        velocity_y_valid = velocity_y_valid/time_disp

        velocity_x = np.zeros(len(xctr))
        velocity_y = np.zeros(len(yctr))
        velocity_x[valid == 1] = velocity_x_valid
        velocity_y[valid == 1] = velocity_y_valid
        

        headings = np.zeros(len(xctr))
        headings[valid == 1] = valid_heading

        cur_traj = np.stack([xctr, yctr, zctr, xlen, ylen, zlen, headings, velocity_x, velocity_y, valid], axis=1) # (num_timestamp, 10)



        # plt.plot(xctr[0], yctr[0], '*', color='yellow', markersize=10, label="Start Point")
        # plt.plot(xctr, yctr, 'o-', label="Original Path", color='orange', markersize=1)
        # plt.show()

        # plt.plot(headings)
        # plt.title("heading from derivative" + key)
        # plt.show()  



        # set object_id
        track_infos['object_id'].append(object_id_start)
        object_id_start += 1
        # Passenger_Vehicle_xctr

        # set object_type_class
        if 'Dummy' in key:
            track_infos['object_type_class'].append('Dummy')
        elif 'Vehicle' in key:
            track_infos['object_type_class'].append('Vehicle')
        elif 'VRU' in key:
            track_infos['object_type_class'].append('VRU')
        else:
            track_infos['object_type_class'].append('Other')
            warnings.warn(f"Warning: The key '{key}' does not contain 'Vehicle' or 'VRU'. Classified as 'Other'.")
        
        # set object_type_subclass
        cur_subclass = '_'.join(key.split('_')[:-1])

        if any(subclass in cur_subclass_clean for subclass in subclass_values[1:]):
            track_infos['object_type_subclass'].append(cur_subclass)
        else:
            track_infos['object_type_subclass'].append('Other')
            warnings.warn(f"Warning: The key '{key}' does not contain any of the predefined subclass. Classified as 'Other'.")
        

        if dummy_flag == True:
            track_infos['object_type'].append('TYPE_DUMMY')
        elif cur_subclass_clean in pedestrian_class:
            track_infos['object_type'].append('TYPE_PEDESTRIAN')
        elif cur_subclass_clean in cyclist_class:
            track_infos['object_type'].append('TYPE_CYCLIST')
        elif cur_subclass_clean in vehicle_class:
            track_infos['object_type'].append('TYPE_VEHICLE')
        else:
            print('?')
            print(cur_subclass)
            track_infos['object_type'].append('Other')


        track_infos['trajs'].append(cur_traj)

    track_infos['trajs'] = np.stack(track_infos['trajs'], axis=0)  # (num_objects, num_timestamp, 9)
    return track_infos, object_id_start


In [8]:
with open("../../map/vector_map.pkl", "rb") as f:
    vector_map = pickle.load(f)
def get_polyline_dir(polyline):
    polyline_pre = np.roll(polyline, shift=1, axis=0)
    polyline_pre[0] = polyline[0]
    diff = polyline - polyline_pre
    polyline_dir = diff / np.clip(np.linalg.norm(diff, axis=-1)[:, np.newaxis], a_min=1e-6, a_max=1000000000)
    return polyline_dir
def decode_map_features_from_proto(map_features):
    polylines = []
    map_infos = {}
    polylines = []
    
    for cur_data in map_features:
        if isinstance(cur_data, Lane):
            if cur_data.type == 1:
                # driving
                global_type = 1
            elif cur_data.type == 2:
                # sidework
                global_type = 2
            elif cur_data.type == 3:
                continue
            else:
                global_type = 0
            cur_polyline = np.stack([np.array([mappoint.x, mappoint.y, 0, global_type]) for mappoint in cur_data.polyline], axis=0)
            boundary_polyline = np.stack([np.array([mappoint.x, mappoint.y, 0, 4]) for mappoint in cur_data.boundary], axis=0)
            cur_polyline = np.concatenate((cur_polyline, boundary_polyline), axis=0)
            cur_polyline_dir = get_polyline_dir(cur_polyline[:, 0:3])
            cur_polyline = np.concatenate((cur_polyline[:, 0:3], cur_polyline_dir, cur_polyline[:, 3:]), axis=-1)

        elif isinstance(cur_data, Crosswalk):
            global_type = 3
            cur_polyline = np.stack([np.array([mappoint.x, mappoint.y, 0, global_type]) for mappoint in cur_data.polygon], axis=0)
            cur_polyline_dir = get_polyline_dir(cur_polyline[:, 0:3])
            cur_polyline = np.concatenate((cur_polyline[:, 0:3], cur_polyline_dir, cur_polyline[:, 3:]), axis=-1)
        
        elif isinstance(cur_data, WalkButton):
            continue

        else:
            print(cur_data)
            raise ValueError
        
        polylines.append(cur_polyline)
    try:
        polylines = np.concatenate(polylines, axis=0).astype(np.float32)
    except:
        polylines = np.zeros((0, 7), dtype=np.float32)
        print('Empty polylines: ')
    map_infos['all_polylines'] = polylines  
    return map_infos


In [9]:
def create_info_single_prediction_scenario(info, total_track_infos, start_timestamp, map_infos, prediction_scenario_length = 91, cur_timestamp = 10):
    prediction_info = {}
    prediction_info['scenario_id'] = info['file_id'] + '_' + str(start_timestamp)
    prediction_info['timestamps_seconds'] = info['timestamps_seconds'][start_timestamp: start_timestamp + prediction_scenario_length] 
    # hard code current tine index
    prediction_info['current_time_index'] = cur_timestamp  #10
    
    # a track is considered "track_to_predict" is it is valid over the whole prediction interval
    track_valid = total_track_infos['trajs'][:, start_timestamp: start_timestamp + prediction_scenario_length, -1]
    track_valid_mask = np.all(track_valid != 0, axis=1)
    track_valid_indices = np.where(track_valid_mask)[0]


    if len(track_valid_indices) == 0:
        return None
    
    # note that track index is consistent in the same run file, but start from 0 in each file
    prediction_info['tracks_to_predict'] = {
        'track_index': track_valid_indices
    }

    prediction_info['tracks_to_predict']['object_type_subclass'] = [total_track_infos['object_type_subclass'][cur_idx] for cur_idx in prediction_info['tracks_to_predict']['track_index']]
    prediction_info['tracks_to_predict']['object_type_class'] = [total_track_infos['object_type_class'][cur_idx] for cur_idx in prediction_info['tracks_to_predict']['track_index']]
    prediction_info['tracks_to_predict']['object_type'] = [total_track_infos['object_type'][cur_idx] for cur_idx in prediction_info['tracks_to_predict']['track_index']]
    prediction_info['trajs'] = total_track_infos['trajs'][:, start_timestamp: start_timestamp + prediction_scenario_length, :]
    
    # Define the valid ranges for x and y
    x_min, x_max = -30, 62
    y_min, y_max = -70, 75
    x_out_of_range = np.logical_or(prediction_info['trajs'][..., 0] < x_min, prediction_info['trajs'][..., 0] > x_max)
    y_out_of_range = np.logical_or(prediction_info['trajs'][..., 1] < y_min, prediction_info['trajs'][..., 1] > y_max)
    if np.any(x_out_of_range) or np.any(y_out_of_range):
        return None
  

    track_infos = {}
    track_infos['object_id'] = total_track_infos['object_id']
    track_infos['object_type_subclass'] = total_track_infos['object_type_subclass']
    track_infos['object_type_class'] = total_track_infos['object_type_class']   
    track_infos['object_type'] = total_track_infos['object_type']
    track_infos['trajs'] = total_track_infos['trajs'][:, start_timestamp: start_timestamp + prediction_scenario_length, :]
    prediction_info['track_infos'] = track_infos
    prediction_info['map_infos'] = map_infos

    return prediction_info
   
    



In [13]:
def create_infos_from_csv(csv_file, object_id_start,vector_map):
    df = pd.read_csv(csv_file)
    csv_keygroup = get_object_keygroups(df)
    info = {}
    info['file_id'] = csv_file.split('/')[-1].split('_')[1]
    info['timestamps_seconds'] = list(df['Time'])
    info['prediction_scenarios'] = []    
    total_track_infos, object_id_start = decode_tracks_from_csv(df, object_id_start, csv_file)
    map_infos = decode_map_features_from_proto(vector_map.map_features)
    # prediction_scenario_length = 131 # past 50, current 1, future 80
    prediction_scenario_length = 61
    cur_timestamp = 10
    for start_timestamp in range(len(info['timestamps_seconds']) - prediction_scenario_length + 1):
        prediction_info = create_info_single_prediction_scenario(info, total_track_infos, start_timestamp, map_infos,prediction_scenario_length, cur_timestamp)
        if prediction_info == None:
            continue
        info['prediction_scenarios'].append(prediction_info)    
    info['traj'] = total_track_infos['trajs']
    return info, object_id_start
        


In [14]:
# set training and validation files
# import random
# traj_val_gt_files = glob.glob(os.path.join('/data/tianhui/challenge_validation_second_release', "*.csv"))
# val_files = random.sample(traj_val_gt_files, 5)
# train_files = list(set(traj_val_gt_files) - set(val_files))

traj_val_gt_files = glob.glob(os.path.join('../../Data/validation_GT_second_release', "*.csv"))
traj_ann_gt_files = glob.glob(os.path.join('../../Data/annotation', "*.csv"))
traj_ann2_gt_files = glob.glob(os.path.join('../../Data/annotation2', "*.csv"))

# perception_output1 = glob.glob(os.path.join('../../Data/perception_results', "*.csv"))
# perception_output2 = glob.glob(os.path.join('../../Data/perception_results2', "*.csv"))
# 1229, 1256, 448, 478, 875,
# val_files = ['/data/tianhui/challenge_validation_second_release/Run_1229_GT.csv','/data/tianhui/challenge_validation_second_release/Run_1256_GT.csv','/data/tianhui/challenge_validation_second_release/Run_448_GT.csv','/data/tianhui/challenge_validation_second_release/Run_478_GT.csv','/data/tianhui/challenge_validation_second_release/Run_875_GT.csv']
# train_files = list(set(traj_val_gt_files) - set(val_files))
# 179，315，214， 410， 1078
val_files = ['../../Data/validation_GT_second_release/Run_179_GT.csv','../../Data/validation_GT_second_release/Run_315_GT.csv','../../Data/validation_GT_second_release/Run_214_GT.csv','../../Data/validation_GT_second_release/Run_410_GT.csv','../../Data/validation_GT_second_release/Run_1078_GT.csv','../../Data/validation_GT_second_release/Run_91_GT.csv','../../Data/validation_GT_second_release/Run_448_GT.csv',\
             '../../Data/validation_GT_second_release/Run_875_GT.csv', '../../Data/validation_GT_second_release/Run_48_GT.csv','../../Data/validation_GT_second_release/Run_363_GT.csv']

# train_files = list(set(traj_val_gt_files + traj_ann_gt_files + traj_ann2_gt_files + perception_output1 + perception_output2) - set(val_files))
train_files = list(set(traj_val_gt_files + traj_ann_gt_files + traj_ann2_gt_files) - set(val_files))

In [ ]:
# using only validation GT files for training
traj_val_gt_files = glob.glob(os.path.join('../../Data/validation_GT_second_release', "*.csv"))
traj_ann_gt_files = glob.glob(os.path.join('../../Data/annotation', "*.csv"))

val_files = ['../../Data/validation_GT_second_release/Run_179_GT.csv','../../Data/validation_GT_second_release/Run_315_GT.csv','../../Data/validation_GT_second_release/Run_214_GT.csv','../../Data/validation_GT_second_release/Run_410_GT.csv','../../Data/validation_GT_second_release/Run_1078_GT.csv','../../Data/validation_GT_second_release/Run_91_GT.csv','../../Data/validation_GT_second_release/Run_448_GT.csv',\
             '../../Data/validation_GT_second_release/Run_875_GT.csv', '../../Data/validation_GT_second_release/Run_48_GT.csv','../../Data/validation_GT_second_release/Run_363_GT.csv']

train_files = list(set(traj_val_gt_files) - set(val_files))

In [15]:
output_path = '/data/tianhui/Challenge/val_ann12_h1sf5s/processed_scenarios_training'
object_id_start = 0
for traj_train_gt_file in train_files:
    info, object_id_start = create_infos_from_csv(traj_train_gt_file,object_id_start,vector_map)
    for prediction_senario in info['prediction_scenarios']:
        scenario_id = prediction_senario['scenario_id']
        output_file = os.path.join(output_path, f'sample_{scenario_id}.pkl')
        with open(output_file, 'wb') as f:
            pickle.dump(prediction_senario, f)

Optimization failed
-------all static--------
Optimization failed
-------all static--------
Optimization failed
-------all static--------


In [16]:
# get validation split
output_path = '/data/tianhui/Challenge/val_ann12_h1sf5s/processed_scenarios_validation'
object_id_start = 0
for traj_val_gt_file in val_files:
    info, object_id_start = create_infos_from_csv(traj_val_gt_file,object_id_start,vector_map)
    for prediction_senario in info['prediction_scenarios']:
        scenario_id = prediction_senario['scenario_id']
        output_file = os.path.join(output_path, f'sample_{scenario_id}.pkl')
        with open(output_file, 'wb') as f:
            pickle.dump(prediction_senario, f)

In [17]:
pkl_path = '/data/tianhui/Challenge/val_ann12_h1sf5s/processed_scenarios_training'
traj_pkls = glob.glob(os.path.join(pkl_path, "*.pkl"))
val_info = []
for pkl in traj_pkls:
    with open(pkl, 'rb') as f:
        predict_dict = pickle.load(f)
        info = {}
        info['scenario_id'] = predict_dict['scenario_id']
        info['timestamps_seconds'] = predict_dict['timestamps_seconds']
        info['current_time_index'] = predict_dict['current_time_index']
        info['tracks_to_predict'] = predict_dict['tracks_to_predict']   
        val_info.append(info)
output_path = '/data/tianhui/Challenge/val_ann12_h1sf5s'
train_filename = os.path.join(output_path, 'processed_scenarios_training_infos.pkl')
with open(train_filename, 'wb') as f:
    pickle.dump(val_info, f)

In [18]:
pkl_path = '/data/tianhui/Challenge/val_ann12_h1sf5s/processed_scenarios_validation'
traj_pkls = glob.glob(os.path.join(pkl_path, "*.pkl"))
val_info = []
for pkl in traj_pkls:
    with open(pkl, 'rb') as f:
        predict_dict = pickle.load(f)
        info = {}
        info['scenario_id'] = predict_dict['scenario_id']
        info['timestamps_seconds'] = predict_dict['timestamps_seconds']
        info['current_time_index'] = predict_dict['current_time_index']
        info['tracks_to_predict'] = predict_dict['tracks_to_predict']   
        val_info.append(info)
output_path = '/data/tianhui/Challenge/val_ann12_h1sf5s'
val_filename = os.path.join(output_path, 'processed_scenarios_validation_infos.pkl')
with open(val_filename, 'wb') as f:
    pickle.dump(val_info, f)